# Imports

In [1]:
%load_ext cudf.pandas

In [2]:
import numpy as np
import os

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import time

In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath("../.."))
%load_ext ElasticNotebook

# Load Data

In [3]:
### cell 0 ###

train_df = pd.read_csv(
    "/home/dias-benchmarks/notebooks/lextoumbourou/feedback3-eda-hf-custom-trainer-sift/input/feedback-prize-english-language-learning/train.csv"
)
test_df = pd.read_csv(
    "/home/dias-benchmarks/notebooks/lextoumbourou/feedback3-eda-hf-custom-trainer-sift/input/feedback-prize-english-language-learning/test.csv"
)

# -- STEFANOS -- Replicate Data

In [4]:
### cell 1 ###

factor = 10
if (
    "IREWR_LESS_REPLICATION" in os.environ
    and os.environ["IREWR_LESS_REPLICATION"] == "True"
):
    factor = factor // 5
train_df = pd.concat([train_df] * factor)
test_df = pd.concat([test_df] * factor)
train_df.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 39110 entries, 0 to 3910
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   text_id      39110 non-null  object
 1   full_text    39110 non-null  object
 2   cohesion     39110 non-null  float64
 3   syntax       39110 non-null  float64
 4   vocabulary   39110 non-null  float64
 5   phraseology  39110 non-null  float64
 6   grammar      39110 non-null  float64
 7   conventions  39110 non-null  float64
dtypes: float64(6), object(2)
memory usage: 89.9+ MB


Let's see a row from each dataset.

In [5]:
train_df.head()

,text_id,full_text,cohesion,syntax,vocabulary,phraseology,grammar,conventions
0,0016926B079C,I think that students would benefit from learn...,3.5,3.5,3.0,3.0,4.0,3.0
1,0022683E9EA5,When a problem is a change you have to let it ...,2.5,2.5,3.0,2.0,2.0,2.5
2,00299B378633,"Dear, Principal\n\nIf u change the school poli...",3.0,3.5,3.0,3.0,3.0,2.5
3,003885A45F42,The best time in life is when you become yours...,4.5,4.5,4.5,4.5,4.0,5.0
4,0049B1DF5CCC,Small act of kindness can impact in other peop...,2.5,3.0,3.0,3.0,2.5,2.5


In [6]:
test_df.head()

,text_id,full_text
0,0000C359D63E,when a person has no experience on a job their...
1,000BAD50D026,Do you think students would benefit from being...
2,00367BB2546B,"Thomas Jefferson once states that ""it is wonde..."
0,0000C359D63E,when a person has no experience on a job their...
1,000BAD50D026,Do you think students would benefit from being...


Then the size of each dataset.

In [7]:
len(train_df), len(test_df)

(39110, 30)

In [8]:
LABEL_COLUMNS = [
    "cohesion",
    "syntax",
    "vocabulary",
    "phraseology",
    "grammar",
    "conventions",
]

# Text Examples

## Random Examples

In [9]:
texts = train_df.sample(frac=1, random_state=420).head(4)

## Lowest Scoring Examples

In [10]:
train_df["total_score"] = train_df[LABEL_COLUMNS].sum(axis=1)
lowest_df = train_df.sort_values("total_score").head(4)

## Highest Scoring Examples

In [11]:
train_df["total_score"] = train_df[LABEL_COLUMNS].sum(axis=1)
highest_df = train_df.sort_values("total_score", ascending=False).head(4)

# Text Overview

## Word Count

In [12]:
%%time
train_df["word_count"] = train_df.full_text.str.split().list.len()

column_view::get_data: Unsupported type: 24
CPU times: user 94 ms, sys: 82.9 ms, total: 177 ms
Wall time: 126 ms
